# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR² dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading

Load Croissant metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List all available record sets, referencing them by their Croissant `@id`. Then review their fields and columns by `@id` as well.

In [ ]:
# Inspect record set structures and collect their @ids
from pprint import pprint

record_sets = []  # list of @ids
if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        print(f"RecordSet @id: {record_set.id}")
        record_sets.append(record_set.id)
        # List fields
        if hasattr(record_set, 'fields'):
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - Field @id: {field.id}	name: {getattr(field, 'name', '')}")
        # List columns
        if hasattr(record_set, 'columns'):
            print("  Columns:")
            for col in record_set.columns:
                print(f"    - Column @id: {col.id}	name: {getattr(col, 'name', '')}")
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction

Let's load records from available record sets using their `@id`. The schema for this dataset may define experimental data, samples, or results as separate record sets.

In [ ]:
# Attempt to list all available record sets
if not record_sets:
    # Try to find record sets by scanning metadata
    record_sets = [s.id for s in getattr(metadata, 'record_sets', [])]

print("Dataset record sets by @id:")
for rid in record_sets:
    print(f"  - {rid}")

# Prepare dictionary of DataFrame per record set
dataframes = {}

# Load records for each record set into a DataFrame
for recset_id in record_sets:
    print(f"\nLoading records for record_set {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded {len(df)} rows with columns:", list(df.columns))

# Show head of first record set if available
if record_sets:
    sample_set = record_sets[0]
    print(f"\nSample of {sample_set} (", len(dataframes[sample_set]), "records):")
    display(dataframes[sample_set].head())

## 4. Exploratory Data Analysis (EDA)

Explore the main table: filter, normalize, and group by fields, always referencing fields by their `@id` as from the prior overview. Choose numeric and categorical fields for demonstration.

In [ ]:
import numpy as np

main_set = record_sets[0] if record_sets else None
df = dataframes[main_set] if main_set else None

if df is not None and not df.empty:
    # Try to auto-select a numeric column by first numeric dtype
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if numeric_cols.any():
        numeric_field_id = numeric_cols[0]
    else:
        # Try to parse some likely numeric columns by name
        likely_numeric = [c for c in df.columns if 'MW' in c or 'abundance' in c or 'count' in c or 'score' in c]
        if likely_numeric:
            numeric_field_id = likely_numeric[0]
            # Coerce to numeric
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        else:
            numeric_field_id = df.columns[0]

    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].std() else 1
    
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")
    
    norm_field = numeric_field_id + '_normalized'
    # Normalize
    filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_field]].head())

    # Try to group by a likely categorical field
    group_candidates = [col for col in df.columns if col.lower() in ('accession', 'description', 'sample', 'protein_id', 'sample_id') or df[col].dtype==object]
    group_field = None
    for gc in group_candidates:
        if gc != numeric_field_id:
            group_field = gc
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"\nGrouped means by {group_field} (top 5):")
        print(grouped_df.head())
else:
    print('No available DataFrame for EDA.')

## 5. Visualization

Simple visualization: histogram of the main numeric field and a bar plot of grouped means, referencing fields by their `@id` where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field is not None:
        # Top categories
        gmeans = grouped_df.head(10)
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field_id, data=gmeans)
        plt.title(f"Mean {numeric_field_id} by {group_field} (Top 10)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion

We have loaded and explored the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library:
- Data and field discovery are fully ID-based using Croissant schema
- Main numeric columns can be filtered, normalized, and grouped for downstream EDA
- Data pipelines are reproducible and precisely reference record sets and fields by Croissant `@id`

**Next steps** might include domain-specific outlier analysis, protein annotation analysis, or merging with external bioinformatics datasets.